# Chapter 1 — Introduction to Large Language Models
### Practice Notebook

*Source: Hands-On Large Language Models, Jay Alammar & Maarten Grootendorst (O'Reilly)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter01/Chapter%201%20-%20Introduction%20to%20Language%20Models.ipynb)

---

This is your **practice workspace** for Chapter 1. Every code cell is a stub — implement the solution yourself. The chapter is mostly conceptual, so this notebook adds hands-on exercises for every major idea in the PDF: bag-of-words, dense embeddings, text generation, and the encoder vs decoder distinction.

---

## Table of Contents

- [Part 1: Tokenisation and Bag-of-Words](#part-1-tokenisation-and-bag-of-words)
  - [Exercise 1.1 — Manual Tokenisation](#exercise-11--manual-tokenisation)
  - [Exercise 1.2 — Build a Vocabulary](#exercise-12--build-a-vocabulary)
  - [Exercise 1.3 — Bag-of-Words Encoding](#exercise-13--bag-of-words-encoding)
  - [Exercise 1.4 — The BoW Limitation](#exercise-14--the-bow-limitation)
- [Part 2: Loading and Using Phi-3](#part-2-loading-and-using-phi-3)
  - [Exercise 2.1 — Load Model and Tokenizer](#exercise-21--load-model-and-tokenizer)
  - [Exercise 2.2 — Create a Pipeline](#exercise-22--create-a-pipeline)
  - [Exercise 2.3 — Your First Generation](#exercise-23--your-first-generation)
- [Part 3: Understanding Generation Parameters](#part-3-understanding-generation-parameters)
  - [Exercise 3.1 — Effect of max_new_tokens](#exercise-31--effect-of-max_new_tokens)
  - [Exercise 3.2 — Greedy vs Sampling](#exercise-32--greedy-vs-sampling)
  - [Exercise 3.3 — Instruction Following](#exercise-33--instruction-following)
- [Part 4: Encoder vs Decoder Mental Model](#part-4-encoder-vs-decoder-mental-model)
  - [Exercise 4.1 — Sentence Embeddings](#exercise-41--sentence-embeddings)
  - [Exercise 4.2 — Semantic Similarity vs BoW Similarity](#exercise-42--semantic-similarity-vs-bow-similarity)
  - [Exercise 4.3 — What Each Model Type Can Do](#exercise-43--what-each-model-type-can-do)
- [Part 5: Mini Project — Semantic FAQ Bot](#part-5-mini-project--semantic-faq-bot)

### [OPTIONAL] — Install packages on Colab

Uncomment and run if you are on a cloud environment.

💡 **GPU required.** Go to Runtime → Change runtime type → GPU (T4 on Colab).

In [ ]:
# %%capture
# !pip install transformers==4.41.2 accelerate==0.31.0 sentence-transformers

---
# Part 1: Tokenisation and Bag-of-Words

Before neural networks, text was represented as simple word-count vectors — the **bag-of-words** model. Understanding it builds intuition for *why* dense embeddings (Part 4) were such a breakthrough.

We will implement BoW from scratch using the same two sentences the book uses:
- `"That is a cute dog"`
- `"My cat is cute"`

## Exercise 1.1 — Manual Tokenisation

Tokenisation is the process of splitting text into individual units (tokens). The simplest approach is splitting by whitespace.

**Task:** Write a function `tokenise(sentence)` that splits a sentence into lowercase tokens by whitespace. Then tokenise both sentences and print the result.

**Expected output:**
```
Sentence 1 tokens: ['that', 'is', 'a', 'cute', 'dog']
Sentence 2 tokens: ['my', 'cat', 'is', 'cute']
```

In [ ]:
sentence1 = "That is a cute dog"
sentence2 = "My cat is cute"


def tokenise(sentence: str) -> list[str]:
    """Split sentence into lowercase tokens by whitespace."""
    # YOUR CODE HERE
    # Hint: str.lower() and str.split()
    pass


tokens1 = tokenise(sentence1)
tokens2 = tokenise(sentence2)

print(f"Sentence 1 tokens: {tokens1}")
print(f"Sentence 2 tokens: {tokens2}")

## Exercise 1.2 — Build a Vocabulary

A **vocabulary** is the sorted list of all unique tokens across all sentences. Each word gets a fixed index position — this index is what becomes the vector dimension.

**Task:** Write a function `build_vocab(token_lists)` that takes a list of token lists and returns a sorted list of all unique tokens.

**Expected output:**
```
Vocabulary (7 words): ['a', 'cat', 'cute', 'dog', 'is', 'my', 'that']
```

In [ ]:
def build_vocab(token_lists: list[list[str]]) -> list[str]:
    """Return sorted list of unique tokens across all token lists."""
    # YOUR CODE HERE
    # Hint: flatten the list of lists, use set() to get unique words, then sorted()
    pass


vocab = build_vocab([tokens1, tokens2])
print(f"Vocabulary ({len(vocab)} words): {vocab}")

## Exercise 1.3 — Bag-of-Words Encoding

A BoW vector has one dimension per vocabulary word. Each dimension holds the **count** of how many times that word appears in the sentence.

**Manual dry-run for `"My cat is cute"` against vocab `['a', 'cat', 'cute', 'dog', 'is', 'my', 'that']`:**
```
a     → appears 0 times → 0
cat   → appears 1 time  → 1
cute  → appears 1 time  → 1
dog   → appears 0 times → 0
is    → appears 1 time  → 1
my    → appears 1 time  → 1
that  → appears 0 times → 0

Vector: [0, 1, 1, 0, 1, 1, 0]
```

**Task:** Write `bow_encode(tokens, vocab)` that returns the count vector for a tokenised sentence. Then encode both sentences and print them.

In [ ]:
def bow_encode(tokens: list[str], vocab: list[str]) -> list[int]:
    """Return a bag-of-words count vector for a list of tokens."""
    # YOUR CODE HERE
    # Hint: for each word in vocab, count how many times it appears in tokens
    pass


vec1 = bow_encode(tokens1, vocab)
vec2 = bow_encode(tokens2, vocab)

print(f"Vocab:      {vocab}")
print(f"Sentence 1: {vec1}   ← '{sentence1}'")
print(f"Sentence 2: {vec2}   ← '{sentence2}'")

## Exercise 1.4 — The BoW Limitation

BoW is simple but has a fundamental flaw: it cannot capture semantic similarity. Two sentences with the same meaning but different words will have completely different (and therefore far-apart) vectors.

**Task:**
1. Add two semantically similar sentences to a new vocabulary
2. Encode them both using BoW
3. Compute their cosine similarity manually using the formula:

$$\text{cosine\_sim}(A, B) = \frac{\sum_i A_i \cdot B_i}{\sqrt{\sum_i A_i^2} \cdot \sqrt{\sum_i B_i^2}}$$

4. Observe that the similarity is low (or even 0) despite the sentences meaning the same thing.

**Test sentences:**
- `"The dog sprinted through the park"`
- `"A puppy ran across the garden"`

**Expected observation:** cosine similarity ≈ 0.0 (no shared words = orthogonal vectors)

In [ ]:
import math

sent_a = "The dog sprinted through the park"
sent_b = "A puppy ran across the garden"

# YOUR CODE HERE
# Step 1: tokenise both sentences
# Step 2: build a combined vocabulary
# Step 3: encode both with bow_encode
# Step 4: compute cosine similarity

def cosine_sim(vec_a: list[int], vec_b: list[int]) -> float:
    """Compute cosine similarity between two count vectors."""
    # YOUR CODE HERE
    # Hint: dot product divided by product of norms
    pass


# Print similarity
# print(f"BoW cosine similarity: {score:.4f}")
# print("Observation: two sentences with the same meaning score near 0")
# print("This is WHY dense embeddings were invented.")

---
# Part 2: Loading and Using Phi-3

This part mirrors the original book notebook exactly. We load Microsoft's **Phi-3-mini** — a 3.8B parameter instruction-following model that runs on a single T4 GPU (~8GB VRAM).

Phi-3 is a **decoder-only** model: it takes a prompt (text in) and generates a completion (text out), token by token.

## Exercise 2.1 — Load Model and Tokenizer

Two objects are needed separately:
- The **model** — the neural network with billions of parameters
- The **tokenizer** — converts text ↔ token IDs

**Task:** Load both from `"microsoft/Phi-3-mini-4k-instruct"`. Use `device_map="cuda"` to put the model on GPU, `torch_dtype="auto"` for automatic precision, and `trust_remote_code=False`.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# YOUR CODE HERE
# model = AutoModelForCausalLM.from_pretrained(
#     "microsoft/Phi-3-mini-4k-instruct",
#     device_map=...,
#     torch_dtype=...,
#     trust_remote_code=...,
# )
# tokenizer = AutoTokenizer.from_pretrained(...)

print("Model loaded successfully.")
print(f"Model device: {next(model.parameters()).device}")

## Exercise 2.2 — Create a Pipeline

Rather than calling the model and tokenizer manually, we wrap them in a **pipeline** — a high-level interface that handles tokenisation, inference, and decoding for us.

**Task:** Create a `pipeline` with task `"text-generation"`. Set:
- `return_full_text=False` — return only the new generated text, not the prompt
- `max_new_tokens=500` — maximum length of generated response
- `do_sample=False` — greedy decoding (deterministic)

In [ ]:
from transformers import pipeline

# YOUR CODE HERE
# generator = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     return_full_text=...,
#     max_new_tokens=...,
#     do_sample=...
# )

print("Pipeline ready.")

## Exercise 2.3 — Your First Generation

Instruction-following models expect input as a list of **messages** — each with a `role` ("user" or "assistant") and `content` (the text).

**Task:** Construct a messages list with one user message asking for a funny joke about chickens. Pass it to the generator and print the output.

**Expected output (approximately):**
```
Why did the chicken join the band? Because it had the drumsticks!
```

In [ ]:
# YOUR CODE HERE
# messages = [
#     {"role": "user", "content": "Create a funny joke about chickens."}
# ]

# output = generator(messages)
# print(output[0]["generated_text"])

---
# Part 3: Understanding Generation Parameters

The pipeline's behaviour is controlled by parameters. This part explores the most important ones so you understand what is actually happening when the model generates text.

## Exercise 3.1 — Effect of `max_new_tokens`

`max_new_tokens` controls the maximum number of tokens the model can generate before it is forced to stop.

**Task:** Run the same prompt with two different `max_new_tokens` values and compare the outputs.

Prompt: `"Explain what a large language model is in detail."`

**Observation to note:** What happens to the response when `max_new_tokens` is very small? Does it end mid-sentence?

In [ ]:
prompt = [{"role": "user", "content": "Explain what a large language model is in detail."}]

# YOUR CODE HERE
# Run the generator with max_new_tokens=30, print the output
# Run the generator with max_new_tokens=300, print the output
# Observe the difference

## Exercise 3.2 — Greedy Decoding vs Sampling

- **Greedy (`do_sample=False`)**: at each step, always pick the single most probable next token. Completely deterministic — same prompt always gives same output.
- **Sampling (`do_sample=True`)**: at each step, randomly sample from the probability distribution over tokens. Non-deterministic — same prompt gives different output each run.

**Task:** Run the same prompt 3 times with each setting and observe whether the outputs differ.

Prompt: `"Tell me a fun fact about space."`

In [ ]:
prompt = [{"role": "user", "content": "Tell me a fun fact about space."}]

# YOUR CODE HERE
# Run 3 times with do_sample=False — outputs should be identical
print("Greedy (do_sample=False):")
# for i in range(3):
#     ...

print("\nSampling (do_sample=True):")
# Run 3 times with do_sample=True — outputs should vary
# for i in range(3):
#     ...

## Exercise 3.3 — Instruction Following

Instruction-following models are trained to respond differently to different prompt types. The format of your prompt significantly affects the format and quality of the output.

**Task:** Try three different prompt types and print each output. Then write your observation about which type produced the most structured, reliable output.

Prompts to try:
1. A direct question: `"What is the capital of France?"`
2. A task instruction: `"Summarise the following in one sentence: The transformer architecture uses self-attention mechanisms to process sequences in parallel, enabling much faster training than recurrent networks."`
3. A creative task: `"Write a haiku about machine learning."`

In [ ]:
prompts = [
    "What is the capital of France?",
    "Summarise the following in one sentence: The transformer architecture uses self-attention mechanisms to process sequences in parallel, enabling much faster training than recurrent networks.",
    "Write a haiku about machine learning."
]

# YOUR CODE HERE
# For each prompt, wrap in the messages format and call generator
# Print the prompt label and the output

# for prompt_text in prompts:
#     messages = [{"role": "user", "content": prompt_text}]
#     ...

---
# Part 4: Encoder vs Decoder Mental Model

Chapter 1's most important conceptual distinction:

| Model type | Architecture | Input → Output | Examples |
|---|---|---|---|
| **Encoder-only** (representation) | BERT-based | Text → Vector | BERT, sentence-transformers |
| **Decoder-only** (generative) | GPT-based | Text → Text | GPT, Phi-3, Llama |

Encoder models **compress** text into dense vectors. Decoder models **generate** new text.
They are built for different things and cannot be swapped.

## Exercise 4.1 — Sentence Embeddings with a Representation Model

A sentence embedding model (encoder-only) turns a sentence into a fixed-size numeric vector. The vector's position in high-dimensional space encodes the sentence's meaning.

**Task:**
1. Load `all-MiniLM-L6-v2` from sentence-transformers
2. Embed these 5 sentences
3. Print the shape of each embedding

Sentences:
- `"Large language models are trained on vast amounts of text."`
- `"LLMs learn from enormous text datasets."`
- `"My cat loves sleeping in the sun."`
- `"Attention mechanisms allow models to focus on relevant words."`
- `"The stock market fell sharply today."`

**Expected shape per embedding:** `(384,)` — a 384-dimensional vector

In [ ]:
from sentence_transformers import SentenceTransformer

sentences = [
    "Large language models are trained on vast amounts of text.",
    "LLMs learn from enormous text datasets.",
    "My cat loves sleeping in the sun.",
    "Attention mechanisms allow models to focus on relevant words.",
    "The stock market fell sharply today."
]

# YOUR CODE HERE
# encoder = SentenceTransformer('all-MiniLM-L6-v2')
# embeddings = encoder.encode(sentences)
# for sent, emb in zip(sentences, embeddings):
#     print(f"Shape: {emb.shape}  |  Sentence: {sent[:50]}...")

## Exercise 4.2 — Semantic Similarity vs BoW Similarity

In Part 1, BoW gave near-zero similarity for semantically identical sentences with different words. Dense embeddings fix this.

**Task:** Compare BoW cosine similarity vs embedding cosine similarity for the same two sentence pairs:
- Pair A (same meaning, different words): `"The dog sprinted through the park"` vs `"A puppy ran across the garden"`
- Pair B (very different meaning): `"The dog sprinted through the park"` vs `"Stock markets fell sharply today"`

**Expected pattern:**
```
         BoW sim    Embedding sim
Pair A:   ~0.00        ~0.70      ← embeddings capture meaning
Pair B:   ~0.00        ~0.05      ← embeddings also know these are unrelated
```

In [ ]:
import numpy as np

pair_a = ("The dog sprinted through the park", "A puppy ran across the garden")
pair_b = ("The dog sprinted through the park", "Stock markets fell sharply today")

# YOUR CODE HERE
# For each pair:
#   1. Compute BoW cosine similarity (reuse functions from Part 1)
#   2. Compute embedding cosine similarity using numpy:
#      np.dot(emb_a, emb_b) / (np.linalg.norm(emb_a) * np.linalg.norm(emb_b))
#   3. Print both scores side by side

# for label, (s1, s2) in [("Pair A", pair_a), ("Pair B", pair_b)]:
#     ...

## Exercise 4.3 — What Each Model Type Can Do

Encoder models produce **vectors**, not text. Decoder models produce **text**, not vectors.
They are not interchangeable.

**Task:**
1. Ask the **decoder (Phi-3)** to produce an embedding — print what it actually returns
2. Ask the **encoder (sentence-transformer)** to generate text — print what it actually returns
3. Write one-line observations about each result in the markdown cell below

In [ ]:
# Task 1: Ask the decoder (generator) for an embedding
print("Asking the decoder for an embedding:")
# YOUR CODE HERE
# messages = [{"role": "user", "content": "Give me a 384-dimensional vector embedding for the word 'cat'"}]
# output = generator(messages)
# print(type(output[0]["generated_text"]))
# print(output[0]["generated_text"][:200])

print("\nAsking the encoder to generate text:")
# Task 2: Ask the encoder (sentence transformer) to generate text
# YOUR CODE HERE
# result = encoder.encode("Tell me a joke about chickens")
# print(type(result))
# print(f"Shape: {result.shape}")
# print("Observation: encoder always returns a vector, never text")

**Your observations:**

- Decoder asked for an embedding: *(write what you got)*
- Encoder asked to generate text: *(write what you got)*
- Why can't they do each other's jobs? *(one sentence)*

---
# Part 5: Mini Project — Semantic FAQ Bot

This project ties together everything from the chapter:
- Part 1 showed why BoW fails for meaning-based search
- Part 4 showed how embeddings capture meaning
- Part 2 showed how to generate text with a decoder

Now combine them: use an **encoder** to find the most relevant FAQ answer, then use a **decoder** to rephrase it in a friendly tone.

This is the simplest possible **Retrieval-Augmented Generation (RAG)** pipeline.

## The FAQ Data

We have a small FAQ about large language models. Given any user question, we want to:
1. Find the most relevant FAQ entry using semantic similarity (encoder)
2. Rephrase the answer in a friendly, conversational tone (decoder)

In [ ]:
# FAQ: list of (question, answer) pairs
faq = [
    (
        "What is a large language model?",
        "A large language model is a neural network trained on vast amounts of text data to understand and generate human language."
    ),
    (
        "How much GPU memory do I need to run an LLM?",
        "Most small LLMs (3-7B parameters) require 8-16GB of GPU VRAM. Larger models may need 40GB or more."
    ),
    (
        "What is the difference between BERT and GPT?",
        "BERT is an encoder-only model used for understanding and embedding text. GPT is a decoder-only model used for generating text."
    ),
    (
        "What is fine-tuning?",
        "Fine-tuning is the process of continuing to train a pre-trained model on a smaller, task-specific dataset to improve its performance on that task."
    ),
    (
        "What is tokenisation?",
        "Tokenisation is the process of splitting text into smaller units called tokens, which are the basic input units that language models process."
    ),
]

faq_questions = [q for q, _ in faq]
faq_answers = [a for _, a in faq]

print(f"FAQ loaded: {len(faq)} entries")

### Step 1 — Embed All FAQ Questions

**Task:** Use the sentence-transformer encoder to embed all 5 FAQ questions. Store them as `faq_embeddings`.

Print the shape — should be `(5, 384)`.

In [ ]:
# YOUR CODE HERE
# faq_embeddings = encoder.encode(faq_questions)
# print(f"FAQ embeddings shape: {faq_embeddings.shape}")

### Step 2 — Find the Best Matching FAQ Entry

Given a user question, embed it and find the FAQ question with the highest cosine similarity.

**Task:** Write `find_best_match(user_question, faq_embeddings)` that:
1. Embeds the user question
2. Computes cosine similarity with all FAQ embeddings
3. Returns the index of the best-matching FAQ entry

**Test question:** `"How do encoder and decoder models differ?"`

**Expected match:** FAQ entry 3 (BERT vs GPT question)

In [ ]:
import numpy as np


def find_best_match(user_question: str, faq_embs: np.ndarray) -> int:
    """Return the index of the FAQ entry most similar to the user question."""
    # YOUR CODE HERE
    # 1. Embed user_question with encoder
    # 2. Compute cosine similarity with each FAQ embedding
    # 3. Return index of highest similarity
    # Hint: np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)) for each FAQ row
    pass


user_question = "How do encoder and decoder models differ?"
# best_idx = find_best_match(user_question, faq_embeddings)
# print(f"Best match FAQ question: {faq_questions[best_idx]}")
# print(f"Matched answer: {faq_answers[best_idx]}")

### Step 3 — Rephrase with the Decoder

Pass the matched FAQ answer to Phi-3 and ask it to rephrase it in a friendly, conversational tone.

**Task:** Build a prompt that includes the retrieved answer and asks Phi-3 to rephrase it. Print the final output.

**Prompt template to use:**
```
A user asked: "{user_question}"
Here is the factual answer: "{retrieved_answer}"
Please rephrase this answer in a friendly and conversational tone in 1-2 sentences.
```

In [ ]:
# YOUR CODE HERE
# retrieved_answer = faq_answers[best_idx]

# prompt_text = f"""A user asked: "{user_question}"
# Here is the factual answer: "{retrieved_answer}"
# Please rephrase this answer in a friendly and conversational tone in 1-2 sentences."""

# messages = [{"role": "user", "content": prompt_text}]
# output = generator(messages)
# print("User question:", user_question)
# print("\nRetrieved FAQ answer:", retrieved_answer)
# print("\nFinal rephrased answer:")
# print(output[0]["generated_text"])

### Wrap It All Up

**Task:** Put Steps 1–3 together in a single `answer_question(user_question)` function. Test it on at least 3 different questions.

Test questions:
- `"What do I need to run an LLM at home?"`
- `"Can you explain what tokens are?"`
- `"What does it mean to fine-tune a model?"`

In [ ]:
def answer_question(user_question: str) -> str:
    """Find the best FAQ match and rephrase the answer using the generation model."""
    # YOUR CODE HERE
    pass


test_questions = [
    "What do I need to run an LLM at home?",
    "Can you explain what tokens are?",
    "What does it mean to fine-tune a model?"
]

# for q in test_questions:
#     print(f"Q: {q}")
#     print(f"A: {answer_question(q)}")
#     print()

---
## Chapter 1 Summary

You have now implemented the key ideas from Chapter 1 from scratch:

| Concept | What you built | Key insight |
|---|---|---|
| Bag-of-Words | Tokeniser, vocab builder, encoder, cosine similarity | Word counts cannot capture meaning |
| Text generation | Phi-3 pipeline, greedy vs sampling, prompt types | Decoders produce text token by token |
| Dense embeddings | Sentence-transformer encoder, similarity comparison | Embeddings capture semantics BoW cannot |
| Encoder vs decoder | Tried to swap them | They are built for different things |
| Mini RAG | Retrieve by embedding → rephrase by generation | Encoder + decoder together = powerful |

The progression from BoW → embeddings → generation → retrieval+generation mirrors exactly how the field evolved from 2013 to 2023.